In [ ]:
eval_dataset = "GSM8K"
model = "Qwen2.5-7B-Instruct"
eval_type = "all"
prompt_method = ["cot0shot", "direct"]
use_template = True
file_name = "gsm8k_bottom_0.000000_alpaca_cleaned_no_safety_seed_0.jsonl"
output_dir = (f"/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/"
f"out/eval_{eval_dataset}/{model}/full_model/eval_{eval_type}/"
f"prompt_{prompt_method}/add_template_{use_template}/")

## eval dataset selection

In [ ]:
import json, os, random
from typing import Dict, List

# ==== 配置 ====
seed = 42
eval_dataset = "GSM8K"
model = "Qwen2.5-7B-Instruct"
eval_type = "all"
use_template = False
file_name = "gsm8k_bottom_0.000000_alpaca_cleaned_no_safety_seed_0.jsonl"

base_dir = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out"
cot0shot_file = os.path.join(
    base_dir,
    f"eval_{eval_dataset}/{model}/full_model/eval_{eval_type}/prompt_cot0shot/add_template_{use_template}/{file_name}"
)
direct_file = os.path.join(
    base_dir,
    f"eval_{eval_dataset}/{model}/full_model/eval_{eval_type}/prompt_direct/add_template_{use_template}/{file_name}"
)
cot0shot_goldreason_file = os.path.join(
    base_dir,
    f"eval_{eval_dataset}/{model}/full_model/eval_{eval_type}/prompt_cot0shot_goldreason/add_template_{use_template}/{file_name}"
)
cot4shot_file = os.path.join(
    base_dir,
    f"eval_{eval_dataset}/{model}/full_model/eval_{eval_type}/prompt_cot4shot/add_template_{use_template}/{file_name}"
)

# 输出目录
out_dir_calibration = f"/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/{model}/calibration_datasets"
out_dir_eval = f"/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/data/GSM8K_eval_build/{model}/eval_datasets"
os.makedirs(out_dir_calibration, exist_ok=True)
os.makedirs(out_dir_eval, exist_ok=True)

# 目标规模
TARGET_PER_BUCKET = 300     # eval A/B 各取 300
CALIB_TOTAL       = 120     # calibration 总数（不分 A/B）

# ==== 工具函数 ====
def load_jsonl_by_id(path: str) -> Dict[str, dict]:
    data = {}
    with open(path, "r") as f:
        for line in f:
            obj = json.loads(line.strip())
            qid = obj.get("id")
            if qid:
                data[qid] = obj
    return data

def save_jsonl(path: str, rows: List[dict]):
    with open(path, "w") as f:
        for rec in rows:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def sample_ids(ids: List[str], k: int, rng: random.Random) -> List[str]:
    if not ids:
        return []
    k = min(k, len(ids))
    return rng.sample(ids, k)

# ==== 加载 ====
cot0 = load_jsonl_by_id(cot0shot_file)
direct = load_jsonl_by_id(direct_file)
cot0_gr = load_jsonl_by_id(cot0shot_goldreason_file)
cot4 = load_jsonl_by_id(cot4shot_file)

shared4_ids = set(cot0) & set(direct) & set(cot0_gr) & set(cot4)
print(f"[LOAD] shared across 4 prompts: {len(shared4_ids)}")

# ==== 构造 A/B 桶（eval 用）====
bucket_A_ids = [q for q in shared4_ids if cot0[q].get("correct") and direct[q].get("correct")]
bucket_B_ids = [q for q in shared4_ids if cot0[q].get("correct") and (not direct[q].get("correct"))]

rng = random.Random(seed)

# ==== Step 1: eval 先取 A/B 各 300 ====
A_eval_ids = sample_ids(bucket_A_ids, TARGET_PER_BUCKET, rng)
B_eval_ids = sample_ids(bucket_B_ids, TARGET_PER_BUCKET, rng)
eval_ids   = set(A_eval_ids) | set(B_eval_ids)

save_jsonl(os.path.join(out_dir_eval, "cot_correct_direct_correct_selected_samples.jsonl"),
           [cot0[q] for q in A_eval_ids])
save_jsonl(os.path.join(out_dir_eval, "cot_correct_direct_incorrect_selected_samples.jsonl"),
           [cot0[q] for q in B_eval_ids])
save_jsonl(os.path.join(out_dir_eval, "selected_samples.jsonl"),
           [cot0[q] for q in (A_eval_ids + B_eval_ids)])
print(f"✅ Saved eval datasets: A={len(A_eval_ids)}, B={len(B_eval_ids)}")

# ==== Step 2: calibration 从剩余里抽一套 ID（不分 A/B，总共 240）====
remain_ids = list(shared4_ids - eval_ids)
calib_ids = sample_ids(remain_ids, CALIB_TOTAL, rng)

# 保存 ID 清单
with open(os.path.join(out_dir_calibration, "calibration_ids.json"), "w") as f:
    json.dump({"calibration_ids": calib_ids, "total": len(calib_ids)}, f, indent=2, ensure_ascii=False)

# ==== Step 3: 用同一批 ID 从 4 个 prompt 文件里抽记录 ====
def rows_from_ids(store: Dict[str, dict], ids: List[str]) -> List[dict]:
    return [store[q] for q in ids if q in store]

PROMPTS = {
    "cot0shot": cot0,
    "direct": direct,
    "cot0shot_goldreason": cot0_gr,
    "cot4shot": cot4,
}

for name, store in PROMPTS.items():
    rows = rows_from_ids(store, calib_ids)
    path = os.path.join(out_dir_calibration, f"calibration_{name}_shared_{len(rows)}.jsonl")
    save_jsonl(path, rows)
    print(f"✅ Saved calibration {name}: {len(rows)}")


[LOAD] shared across 4 prompts: 8792
✅ Saved eval datasets: A=300, B=300
✅ Saved calibration cot0shot: 120
✅ Saved calibration direct: 120
✅ Saved calibration cot0shot_goldreason: 120
✅ Saved calibration cot4shot: 120
